In [25]:
import pandas as pd
from pathlib import Path
import sys
import importlib
importlib.reload(prep)

# 1 Path
DATA_DIR = Path.cwd().parent / "data"

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import src.preprocessing as prep    

# 2 Extract
RAW_4G = pd.read_csv(DATA_DIR / "public_github_4g_lte.csv")
RAW_5G = pd.read_csv(DATA_DIR / "public_github_5g_nr.csv")

# 3 Inspections    

summary_raw_4g = prep.diagnose(RAW_4G)

summary_raw_5g = prep.diagnose(RAW_5G)


# 4 data processing 

df_4g = prep.date_time(RAW_4G)
df_5g = prep.date_time(RAW_5G)

KEY = ["timestamp", "site_id", "sector_id"]

prep.validate_non_negative(
    df_4g,
    prep.VOLUMETRIC_COLS
)
#Verify unique key value
#df_4g.duplicated(subset  = KEY).sum()

df_4g, rejected_4g = prep.remove_invalid_rows(
    df_4g,
    prep.VOLUMETRIC_COLS_4G,
    prep.PERCENTAGE_COLS_4G
)

df_5g, rejected_5g = prep.remove_invalid_rows(
    df_5g,
    prep.VOLUMETRIC_COLS_5G,
    prep.PERCENTAGE_COLS_5G
    
)

# Keep only the hours between 08:00 and 20:00 to analyze network indicators during the most representative period of daily traffic.
df_4g = df_4g[
    df_4g["timestamp"].dt.hour.between(8, 20)]

df_5g = df_5g[
        df_5g["timestamp"].dt.hour.between(8, 20)]


In [26]:
indicator_results_4g = prep.evaluate_indicators(
    df_4g,
    prep.THRESHOLDS_4G,
    prep.GROUP_COLS_4G
)

display(indicator_results_4g.head())

indicator_results_5g = prep.evaluate_indicators(
    df_5g,
    prep.THRESHOLDS_5G,
    prep.GROUP_COLS_5G
)

display(indicator_results_5g.head())

,site_id,sector_id,indicator,threshold,required_pass_ratio,total_samples,valid_samples,valid_ratio,pass_ratio,indicator_status
0,SITE_018,3,accessibility_pct,99.0,0.8,182,181,0.994505,1.0,PASS
1,SITE_018,3,availability_pct,99.0,0.8,182,181,0.994505,1.0,PASS
2,SITE_018,3,csfb_preparation_success_pct,99.0,0.8,182,173,0.950549,1.0,PASS
3,SITE_018,3,erab_success_pct,99.0,0.8,182,181,0.994505,1.0,PASS
4,SITE_018,3,handover_success_pct,97.0,0.8,182,181,0.994505,1.0,PASS


,site_id,sector_id,band_category,indicator,threshold,required_pass_ratio,total_samples,valid_samples,valid_ratio,pass_ratio,indicator_status
0,SITE_012,1,MID,sgnb_endc_drop_rate_pct,2,0.8,182,182,1.000000,1.0,PASS
1,SITE_012,1,MID,inter_sa_handover_success_pct,97,0.8,182,177,0.972527,1.0,PASS
2,SITE_012,1,MID,sa_rrc_success_pct,99,0.8,182,182,1.000000,1.0,PASS
3,SITE_012,1,MID,sa_qos_flow_success_pct,99,0.8,182,182,1.000000,1.0,PASS
4,SITE_012,1,MID,intra_sa_handover_success_pct,97,0.8,182,182,1.000000,1.0,PASS


In [28]:
site_indicators_4g = prep.summarize_site_indicators(
    indicator_results_4g
)

site_indicators_5g = prep.summarize_site_indicators(
    indicator_results_5g
)
display(site_indicators_4g.head(20))


,site_id,indicator,indicator_status
0,SITE_001,accessibility_pct,PASS
1,SITE_001,availability_pct,PASS
2,SITE_001,avg_total_users,PASS
3,SITE_001,avg_users,PASS
4,SITE_001,avg_volte_users,PASS
5,SITE_001,csfb_preparation_success_pct,PASS
6,SITE_001,dl_buffer_latency_ms,PASS
7,SITE_001,dl_throughput_kbps,PASS
8,SITE_001,dl_volume_mb,PASS
9,SITE_001,erab_success_pct,PASS


In [31]:
site_summary_4g = prep.classify_sites(
    site_indicators_4g
)

site_summary_5g = prep.classify_sites(
    site_indicators_5g
)

display(site_summary_4g)

,site_id,total_indicators,no_pass_indicators,insufficient_indicators,no_pass_indicator_ratio,site_classification
0,SITE_001,25,1,0,0.04,PASS
1,SITE_002,25,1,0,0.04,PASS
2,SITE_003,25,1,0,0.04,PASS
3,SITE_004,25,1,0,0.04,PASS
4,SITE_005,25,1,0,0.04,PASS
5,SITE_006,25,1,0,0.04,PASS
6,SITE_007,25,1,0,0.04,PASS
7,SITE_008,25,1,0,0.04,PASS
8,SITE_009,25,1,0,0.04,PASS
9,SITE_010,25,1,0,0.04,PASS


In [33]:
status_map_4g = (
    site_summary_4g
    .set_index("site_id")
    ["site_classification"]
)

df_4g["site_classification"] = (
    df_4g["site_id"]
    .map(status_map_4g)
)

status_map_5g = (
    site_summary_5g
    .set_index("site_id")
    ["site_classification"]
)

df_5g["site_classification"] = (
    df_5g["site_id"]
    .map(status_map_5g)
)

In [34]:
df_4g[
    ["site_id", "site_classification"]
].drop_duplicates()

,site_id,site_classification
162,SITE_018,PASS
165,SITE_017,PASS
168,SITE_016,PASS
171,SITE_015,PASS
174,SITE_014,PASS
177,SITE_013,PASS
180,SITE_012,PASS
183,SITE_011,PASS
186,SITE_010,PASS
189,SITE_009,PASS


In [35]:
pass_4g = (
    df_4g[
        prep.SITE_INFO_COLS_4G
        + ["site_classification"]
    ]
    .drop_duplicates()
)

pass_4g = pass_4g[
    pass_4g["site_classification"] == "PASS"
]

pass_5g = (
    df_5g[
        prep.SITE_INFO_COLS_5G
        + ["site_classification"]
    ]
    .drop_duplicates()
)

pass_5g = pass_5g[
    pass_5g["site_classification"] == "PASS"
]

In [36]:
no_pass_4g = indicator_results_4g[
    indicator_results_4g["indicator_status"]
    == "NO_PASS"
]

no_pass_5g = indicator_results_5g[
    indicator_results_5g["indicator_status"]
    == "NO_PASS"
]

In [37]:
p10_4g = prep.calculate_quantile(
    df_4g,
    prep.GROUP_COLS_4G,
    prep.VOLUMETRIC_COLS_4G,
    0.10
)

p90_4g = prep.calculate_quantile(
    df_4g,
    prep.GROUP_COLS_4G,
    prep.VOLUMETRIC_COLS_4G,
    0.90
)

p10_5g = prep.calculate_quantile(
    df_5g,
    prep.GROUP_COLS_5G,
    prep.VOLUMETRIC_COLS_5G,
    0.10
)

p90_5g = prep.calculate_quantile(
    df_5g,
    prep.GROUP_COLS_5G,
    prep.VOLUMETRIC_COLS_5G,
    0.90
)

In [38]:
OUTPUT_FILE = PROJECT_ROOT / "telecom_pass_nopass_analysis.xlsx"

with pd.ExcelWriter(
    OUTPUT_FILE,
    engine="openpyxl"
) as writer:

    df_4g.to_excel(
        writer,
        sheet_name="4G_COMPLETE",
        index=False
    )

    pass_4g.to_excel(
        writer,
        sheet_name="4G_PASS",
        index=False
    )

    no_pass_4g.to_excel(
        writer,
        sheet_name="4G_NO_PASS",
        index=False
    )

    p10_4g.to_excel(
        writer,
        sheet_name="4G_P10",
        index=False
    )

    p90_4g.to_excel(
        writer,
        sheet_name="4G_P90",
        index=False
    )

    df_5g.to_excel(
        writer,
        sheet_name="5G_COMPLETE",
        index=False
    )

    pass_5g.to_excel(
        writer,
        sheet_name="5G_PASS",
        index=False
    )

    no_pass_5g.to_excel(
        writer,
        sheet_name="5G_NO_PASS",
        index=False
    )

    p10_5g.to_excel(
        writer,
        sheet_name="5G_P10",
        index=False
    )

    p90_5g.to_excel(
        writer,
        sheet_name="5G_P90",
        index=False
    )

ModuleNotFoundError: No module named 'openpyxl'